In [0]:
import pandas as pd
import h5py as hp

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&scope=email%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocs.test%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.photos.readonly%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fpeopleapi.readonly&response_type=code

Enter your authorization code:
··········
Mounted at /content/drive


In [0]:
data = hp.File('/content/drive/My Drive/NN_Project2/SVHN_single_grey1.h5')

In [6]:
print(data.keys())

KeysView(<HDF5 file "SVHN_single_grey1.h5" (mode r+)>)


In [7]:
keys = list(data.keys())
keys

['X_test', 'X_train', 'X_val', 'y_test', 'y_train', 'y_val']

In [0]:
X_test = data['X_test']

In [9]:
print(X_test.shape)


(18000, 32, 32)


In [0]:
X_train = data['X_train']
y_train = data['y_train']
y_test = data['y_test']


In [11]:
print(X_train.shape)
print(y_train.shape)
print(y_test.shape)

(42000, 32, 32)
(42000,)
(18000,)


In [12]:
X_val = data['X_val']
print(X_val.shape)
y_val = data['y_val']
print(y_val.shape)


(60000, 32, 32)
(60000,)


In [0]:
from sklearn.neighbors import KNeighborsClassifier

In [0]:
import numpy as np

In [0]:
X_train_knn = np.reshape(X_train,(42000,1024))
X_test_knn = np.reshape(X_test,(18000,1024))

In [20]:
for k in range(1, 10, 2):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_knn, y_train)
    score = model.score(X_test_knn, y_test)
    print("k=%d, accuracy=%.2f%%" % (k, score * 100))

k=1, accuracy=45.92%
k=3, accuracy=37.84%
k=5, accuracy=32.42%
k=7, accuracy=28.68%
k=9, accuracy=25.98%


This took lot of time to run, so i am not going to run for any other values of k
From the above results K = 1 gives the maximum accuracy. 
So building model with K = 1 and check the accuracy and confusion matrix


In [0]:
model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train_knn, y_train)
predictions = model.predict(X_test_knn)

In [0]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [0]:
cfm = confusion_matrix(y_test,predictions)

In [20]:
print(cfm)

[[ 935   43   51   64   71   65  185   40  170  190]
 [  51 1034  109  122  123   69   68  119   67   66]
 [  63  135  872  131   82   74   65  168   79  134]
 [  67  166  127  610   88  229   82   80  149  121]
 [  85  195   51   71 1039   55  111   35   98   72]
 [  94  103   71  220   54  588  203   48  205  182]
 [ 225   74   51   72  102  154  711   33  295  115]
 [  65  173  146  100   34   38   48 1071   53   80]
 [ 150   62   59  130   86  153  267   44  675  186]
 [ 238   87   89  115   60  113  115   65  192  730]]


In [21]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.47      0.52      0.49      1814
           1       0.50      0.57      0.53      1828
           2       0.54      0.48      0.51      1803
           3       0.37      0.35      0.36      1719
           4       0.60      0.57      0.59      1812
           5       0.38      0.33      0.36      1768
           6       0.38      0.39      0.39      1832
           7       0.63      0.59      0.61      1808
           8       0.34      0.37      0.36      1812
           9       0.39      0.40      0.40      1804

    accuracy                           0.46     18000
   macro avg       0.46      0.46      0.46     18000
weighted avg       0.46      0.46      0.46     18000



Convert y_test and y_train to categorical and normalize X_test and X_train

In [0]:
import tensorflow as tf

In [0]:
y_train_norm = tf.keras.utils.to_categorical(y_train)
y_test_norm = tf.keras.utils.to_categorical(y_test)

In [0]:
X_train_norm = np.array(data['X_train'], dtype=np.float32) /255.0

In [0]:
X_test_norm = np.array(data['X_test'], dtype=np.float32) /255.0

In [20]:
print(X_train_norm.shape)
print(X_test_norm.shape)


(42000, 32, 32)
(18000, 32, 32)


In [21]:
print(y_train_norm.shape)
print(y_test_norm.shape)

(42000, 10)
(18000, 10)


In [22]:
model = tf.keras.models.Sequential()
#model.add(tf.keras.layers.Conv2D(32, kernel_size=(3,3), activation='relu'))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(200, activation='relu'))
model.add(tf.keras.layers.Dense(100, activation='relu'))
#model.add(tf.keras.layers.Dropout(0.25))

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor


In [0]:
model.add(tf.keras.layers.Dense(10, activation='softmax'))

In [0]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [25]:
model.fit(X_train_norm,y_train_norm,          
          validation_data=(X_test_norm,y_test_norm),
          epochs=50,batch_size = 200)

Train on 42000 samples, validate on 18000 samples
Epoch 1/50
42000/42000 [==============================] - 2s 52us/sample - loss: 2.3037 - acc: 0.1159 - val_loss: 2.2538 - val_acc: 0.1555
Epoch 2/50
42000/42000 [==============================] - 2s 43us/sample - loss: 1.9707 - acc: 0.3136 - val_loss: 1.5922 - val_acc: 0.4788
Epoch 3/50
42000/42000 [==============================] - 2s 43us/sample - loss: 1.4528 - acc: 0.5329 - val_loss: 1.2875 - val_acc: 0.5956
Epoch 4/50
42000/42000 [==============================] - 2s 44us/sample - loss: 1.2572 - acc: 0.6072 - val_loss: 1.1652 - val_acc: 0.6341
Epoch 5/50
42000/42000 [==============================] - 2s 46us/sample - loss: 1.1398 - acc: 0.6505 - val_loss: 1.0804 - val_acc: 0.6756
Epoch 6/50
42000/42000 [==============================] - 2s 44us/sample - loss: 1.0730 - acc: 0.6720 - val_loss: 1.0576 - val_acc: 0.6796
Epoch 7/50
42000/42000 [==============================] - 2s 46us/sample - loss: 1.0191 - acc: 0.6896 - val_loss: 1.

In [26]:
print("test score", model.evaluate(X_test_norm,y_test_norm))

18000/18000 [==============================] - 1s 39us/sample - loss: 0.6737 - acc: 0.8095
test score [0.6736580742730035, 0.8095]


In [0]:
model.compile(optimizer='sgd', loss='categorical_crossentropy', metrics=['accuracy'])

In [28]:
model.fit(X_train_norm,y_train_norm,          
          validation_data=(X_test_norm,y_test_norm),
          epochs=50,batch_size = 200)

Train on 42000 samples, validate on 18000 samples
Epoch 1/50
42000/42000 [==============================] - 2s 43us/sample - loss: 0.5011 - acc: 0.8519 - val_loss: 0.6240 - val_acc: 0.8252
Epoch 2/50
42000/42000 [==============================] - 2s 41us/sample - loss: 0.4970 - acc: 0.8525 - val_loss: 0.6243 - val_acc: 0.8247
Epoch 3/50
42000/42000 [==============================] - 2s 40us/sample - loss: 0.4958 - acc: 0.8524 - val_loss: 0.6295 - val_acc: 0.8241
Epoch 4/50
42000/42000 [==============================] - 2s 41us/sample - loss: 0.4947 - acc: 0.8529 - val_loss: 0.6268 - val_acc: 0.8236
Epoch 5/50
42000/42000 [==============================] - 2s 41us/sample - loss: 0.4945 - acc: 0.8543 - val_loss: 0.6226 - val_acc: 0.8256
Epoch 6/50
42000/42000 [==============================] - 2s 41us/sample - loss: 0.4936 - acc: 0.8531 - val_loss: 0.6234 - val_acc: 0.8256
Epoch 7/50
42000/42000 [==============================] - 2s 41us/sample - loss: 0.4923 - acc: 0.8537 - val_loss: 0.

In [68]:
print("test score", model.evaluate(X_test_norm,y_test_norm))

18000/18000 [==============================] - 1s 39us/sample - loss: 0.6195 - acc: 0.8285
test score [0.6195217036273745, 0.8285]


In [41]:
print(X_train_norm.shape)
print(y_train_norm.shape)

(42000, 32, 32)
(42000, 10)


In [0]:
model1 = tf.keras.models.Sequential()
model1.add(tf.keras.layers.Flatten())
#model1.add(tf.keras.layers.Reshape((1024,),input_shape=(32,32,)))
model1.add(tf.keras.layers.BatchNormalization())
model1.add(tf.keras.layers.Dense(200, activation='relu'))
model1.add(tf.keras.layers.Dense(100, activation='relu'))
#model1.add(tf.keras.layers.Dropout(0.25))

In [0]:
model1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [0]:
x_train1 = np.array(data['X_train'], dtype=np.float32)

In [0]:
x_test1 = np.array(data['X_test'], dtype=np.float32)

In [0]:
x_train_new = x_train1.reshape(x_train1.shape[0], 32, 32, 1).astype('float32')
x_test_new = x_test1 .reshape(x_test1 .shape[0], 32, 32, 1).astype('float32')

In [0]:
x_train_new = x_train_new / 255
x_test_new = x_test_new /255


In [0]:
y_train1 = np.array(data['y_train'], dtype=np.float32)
y_test1 = np.array(data['y_test'], dtype=np.float32)

In [0]:
y_train_new = y_train1.reshape(y_train1.shape[0], 1).astype('float32')
y_test_new = y_test1.reshape(y_test1.shape[0],1).astype('float32')

In [0]:
y_train_new = tf.keras.utils.to_categorical(y_train_new,10)
y_test_new = tf.keras.utils.to_categorical(y_test_new,10)

In [67]:
model1.fit(x_train_new,y_train_new,          
          validation_data=(x_test_new,y_test_new),
          epochs=50,batch_size=200)

Train on 42000 samples, validate on 18000 samples
Epoch 1/50
42000/42000 [==============================] - 3s 70us/sample - loss: 2071.8215 - acc: 0.0093 - val_loss: 2073.9729 - val_acc: 0.0104
Epoch 2/50
 5200/42000 [==>...........................] - ETA: 2s - loss: 2080.8695 - acc: 0.0112

KeyboardInterrupt: ignored

I think i have done something wrong while reshaping the data. I am either getting very low accuracy or shape error while using batch normalization. 

Conclusion - 

1. NN model gave better result than KNN, Neural networks are more powerful than traditional ML algorithms. 
2. KNN gave accuracy of 45% while NN model gave accuracy upto 86%. 
3. KNN model took lot of time for training and validation. But NN model is fast and accuracy is also high. 
4. I am sure that addition of batch narmalization must have reduced the traing time.